In [1]:
# Install the project dependencies.
%pip install -q -r ../../requirements.txt


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys
from pathlib import Path
import numpy as np
import plotly.graph_objects as go

from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "scripts"))
from utils import create_pred_input
from simulations.simulation_functions import Surface
RUN = ROOT / "reproduction/weights/2026-05-15_10-52-58_MC_100reps"
FIGURES = ROOT / "paper/Figures"



In [7]:
#function to calculate the surface for the model, benchmark or bias, depending on the element argument. 
################################################## User Input ##################################################
specification = "linear" # "linear", "Burke", "Leirvik", "nonlinear" - true model to be estimated
element = "bias" # "model", "benchmark", "bias" - element to be plotted
model = "Quadratic" # "NN" or "Quadratic" - model that is thrying to estimate the true model

###############################################################################################################

def calculate_surface(specification, element, model):
    T, P = create_pred_input(
        True,
        0,
        0,
        0,
        0,
        two_dim=True,
        only_dims=True
    )
    
    
   
    if model=="NN":
        
        folder = Path(f"../weights/monte_carlo/NN/{specification}")
        
        surface = np.load(next(folder.glob("*.npy")))   # raises StopIteration if folder empty
    else:
        folder = Path(f"../weights/monte_carlo/Quadratic/{specification}")
      
        surface = np.load(next(folder.glob("*.npy")))   # raises StopIteration if folder empty
        
    model_values = np.mean(surface, axis=0).reshape(T.shape)
    benchmark = Surface(T.flatten(), P.flatten(), specification=specification, dynamic=False, time_periods=63).reshape(T.shape)
    

    if element == 'model':
    
        surf = go.Surface(
            x=T, y=P, z=model_values,
            colorscale='Plasma',
            opacity=0.85,
            showscale=False,
            name=f'Model'
            
        )

    elif element == 'benchmark':
        surf= go.Surface(
            x=T, y=P, z=benchmark,
            colorscale='Plasma',
            opacity=0.85,
            showscale=False,
            name=f'Benchmark'
        )
    
    else: 
        bias = np.array(model_values - benchmark)

        zmin, zmax = 0, 1
        ticks = np.linspace(zmin, zmax, 9)           
        ticktext = [f"{t:.1f}" for t in ticks]       

        surf = go.Heatmap(
            x=P[:, 0], y=T[0, :], z=np.sqrt(np.square(bias.T)),
            colorscale='ylorrd', zmin=zmin, zmax=zmax,
            showscale=True,
            colorbar=dict(
                title='Bias',
                thickness=15,
                len=1,
                tickmode='array',
                tickvals=ticks,
                ticktext=ticktext,
            )
        )
    fig=create_figure(element, surf)
    
  
    if element=="bias":
        return fig, bias
    else:
        return fig


# creates specifc layout for each element type (model, benchmark, bias)
def create_figure(element, plot_data):
    
    if element=="model" or element=="benchmark":
        fig = go.Figure(data=plot_data)
        fig.update_layout(
               autosize=False,
        width=500,
        height=600,
        margin=dict(
            l=50,
            r=50,
            b=100,
            t=100,
            pad=4,
        ),

            scene=dict(
                xaxis_title='Temperature (°C)',
                yaxis_title='Precipitation (m)',
                zaxis=dict(autorange=False, title='Δ ln(Growth)',
                range=[-0.1, 0.1] if specification=='linear' else [-0.6, 0.2] if specification=='Burke' else [-1.5, 0.5] if specification=='Leirvik' else [-0.1, 0.2]),
                camera=dict(eye=dict(x=2.35, y=0.006, z=0.4))
            ),
            
            legend=dict(
                bgcolor='rgba(255,255,255,0.7)',
                bordercolor='black',
                borderwidth=1
            ),
            font=dict(
            size=10
        )
        )
        
    else: 
        layout = go.Layout(
        yaxis=dict(autorange='reversed'),
        font=dict(
            size=16
        ))
        fig = go.Figure(data=plot_data, layout=layout)
        fig.update_layout(
        xaxis=dict(title='Precipitation (m)'),
        yaxis=dict(title='Temperature (°C)')
    )



    return fig


In [8]:
for specification in ["linear", "Burke", "Leirvik", "nonlinear"]:
    for element in ["model", "benchmark", "bias"]:
        for model in ["NN"]:
            fig = calculate_surface(specification, element, model)
            if element=="bias":
                fig, bias = fig
            fig.show()
